In [18]:
import kaggle_benchmarks as kbench
import urllib.request
import json
import random
import os

# --------------------------------------------------------------------------------
# 1. LOAD DATA (Root /data JSONL)
# --------------------------------------------------------------------------------
print("Step 1: Downloading authentic EmoBench data...")
base_url = "https://raw.githubusercontent.com/Sahandfer/EmoBench/master/data/"
files = ["EA.jsonl", "EU.jsonl"]
all_samples = []

try:
    for filename in files:
        url = base_url + filename
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            for line in response:
                if line.strip():
                    all_samples.append(json.loads(line.decode('utf-8-sig')))
    
    # Scale to 250 for the final leaderboard run
    random.seed(42)
    random.shuffle(all_samples)
    test_samples = all_samples[:250]
    print(f"Successfully loaded {len(test_samples)} samples.")

except Exception as e:
    print(f"Loading failed: {e}")
    test_samples = []

# --------------------------------------------------------------------------------
# 2. EMOBENCH BENCHMARK TASK (250 Samples)
# --------------------------------------------------------------------------------

@kbench.task(name="EmoBench: Social Intelligence (250)", description="Authentic EmoBench evaluation across 250 samples.")
def emobench_benchmark_250(llm) -> float:
    if not test_samples:
        return 0.0

    correct_count = 0
    total_count = len(test_samples)
    
    print(f"Starting evaluation on {total_count} samples...")

    for idx, sample in enumerate(test_samples):
        # Progress Tracking
        if idx % 50 == 0 and idx > 0:
            print(f"Progress: {idx}/{total_count} samples processed...")

        # Data Extraction
        scenario = sample.get('scenario') or sample.get('situation') or sample.get('context')
        # In this dataset, the 'label' or 'question' often contains the ground truth string
        question_text = sample.get('label') or sample.get('question')
        options = sample.get('choices') or sample.get('options') or []

        if not scenario or len(options) < 4:
            continue

        prompt = (
            f"Social Scenario: {scenario}\n"
            f"Goal: Identify the most emotionally intelligent response.\n\n"
            f"A) {options[0]}\n"
            f"B) {options[1]}\n"
            f"C) {options[2]}\n"
            f"D) {options[3]}\n\n"
            f"Instruction: Output ONLY the letter (A, B, C, or D).\n"
            f"Choice:"
        )

        # Get Prediction
        response = llm.prompt(prompt).upper().strip()
        predicted_letter = next((c for c in response if c in "ABCD"), "")

        # --- ADVANCED GROUND TRUTH MAPPING ---
        # The answer text is usually in 'answer', but fallback to the 'label' field 
        # which you observed contains the correct option text.
        truth_text = sample.get('answer') or sample.get('label') or sample.get('target')
        
        expected_letter = ""
        if str(truth_text).upper() in "ABCD":
            expected_letter = str(truth_text).upper()
        else:
            # Check for exact or normalized string match against options
            for i, opt in enumerate(options):
                if str(truth_text).strip() == str(opt).strip():
                    expected_letter = chr(65 + i)
                    break
        
        if predicted_letter == expected_letter:
            correct_count += 1

    final_score = float(correct_count / total_count)
    print(f"\n--- Final EmoBench Intelligence Score: {final_score:.4f} ---")
    return final_score

# Run
emobench_benchmark_250.run(kbench.llm)

Step 1: Downloading EmoBench JSONL files...
Successfully loaded 5 samples.
  Sample 2: Predicted=C, Expected=C

--- Final Score: 0.2000 ---


BokehModel(combine_events=True, render_bundle={'docs_json': {'4620e80e-bbbf-4855-a960-dc979475c1f3': {'version…